# Full AIDE — test-only trên đúng cohort CommFor cũ

Notebook này **không train lại**. Nó nạp checkpoint Full AIDE tốt nhất đã train trên Tiny-GenImage combined và dự đoán đúng cohort CommFor đã dùng để so sánh AIDE forensic với NPR-ResNet18:

- 100 ảnh real dùng chung;
- 9 generator × 100 ảnh fake = 900 ảnh fake;
- tổng cộng 1.000 ảnh;
- threshold cố định `0.5`, không tune trên CommFor.

Hai SHA-256 được khóa cứng để tránh test nhầm dữ liệu hoặc nhầm weight:

- checkpoint: `62848895b44255d6a0567754c503a05807ed0a29b8046f3c6956a049844e47c3`;
- manifest: `20b04a56d2709d091d51fe585621686b2cd792a373a20ec5068fc6a5be28b9cd`.

## Chuẩn bị trên Kaggle

1. Bật **GPU accelerator**.
2. Tạo Kaggle Dataset từ thư mục `baselines/aide_full/check_point` và attach dataset đó vào notebook. Notebook tự tìm `model_trainable.pt` bằng checksum.
3. Bật Internet để tải `OwensLab/CommunityForensics-Eval` và OpenCLIP ConvNeXt-XXLarge. Nếu đã upload `open_clip_pytorch_model.bin` vào Kaggle Input thì notebook sẽ dùng file local.
4. Nếu dùng Hugging Face thường xuyên, nên tạo Kaggle Secret `HF_TOKEN`.

Kết quả cuối nằm tại `/kaggle/working/aide_full_commfor_exact_eval` và file ZIP tải về nằm tại `/kaggle/working/aide_full_commfor_exact_eval.zip`.

In [ ]:
%pip install -q "open_clip_torch==2.26.1" "datasets>=2.19,<4" "pandas<3.0" "scikit-learn<1.9" "pillow<12.0" "tqdm"

In [ ]:
import os
from pathlib import Path

import torch

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    print("HF_TOKEN not configured; public downloads will be anonymous.")

assert torch.cuda.is_available(), "Kaggle Settings → Accelerator → GPU trước khi chạy."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
CONFIG = {
    "input_root": "/kaggle/input",
    "output_root": "/kaggle/working/aide_full_commfor_exact_eval",

    # None = tự tìm đúng model_trainable.pt bằng SHA-256 trong /kaggle/input.
    "checkpoint_path": None,

    # None = tự tìm open_clip_pytorch_model.bin; nếu không có sẽ tải từ HF.
    "semantic_checkpoint": None,

    "commfor_dataset_name": "OwensLab/CommunityForensics-Eval",
    "commfor_split": "CompEval",
    "commfor_streaming": True,
    "commfor_shuffle_seed": 43,
    "commfor_shuffle_buffer_size": 1000,
    "max_scan_records": 500000,

    "threshold": 0.5,
    "image_size": 256,
    "random_seed": 42,

    # Lưu predictions từng phần; chạy lại cell sẽ bỏ qua các ảnh đã hoàn thành.
    "resume": True,
    "save_every": 25,
}

CONFIG

In [ ]:
import base64
import gzip
import importlib.util
import sys
from pathlib import Path

TRAIN_RUNTIME_GZIP_BASE64 = """..."""
TEST_RUNTIME_GZIP_BASE64 = """H4sIAAAAAAACCsU8a2/bSJLf9Su4PNwtOSMzsmI7iRYawOPYGWMTJ3CcxeB8AkGRLZlritTwYUfj9X+/qupu9oOUbM/s4Qa7sdSP6urq6nq3XNc9zxesZHnM9oo82zhnTZY5x+fvTx12F2VNVKdF7sD/6hvmsKqO5lla3bDEOSlWq7OidOLipijrYDC4ukkrp2zynJVOVkRJRVPmMEeDGd+w+HZdpHnt1GWU5gAIYF+l+WbvA8vPV9GSOVGeDMTajANh36O4dvaHo9FoL6UxuHqTp/UGUGB5lcaVQMRZl+wuLZoKttJUAH6+GSAIXH1vIQbjEs7Fl8u9S1ZdsHr/LaywZmW6YnldTQaDH5z90cipbqISAJQsyhxatfqb6FlEt0w0OYuyWDksim+cYuHksCNnyYAEUV2UFVGF6ZvGhVdRni6QLADemRf1DeBc1CyuCVvn6y/He+PDIz6rCpyW0mk1qOoyjWvYGlCmpvOaAIGA7os0Yw5sAXa4qpy84NRN8+UQ+ktW3RRZ4tQNbwFYqyJh2aBiGSwLBxwMXNcdDGgvYbho6qZkYQhbXCNJozwvamKEajAQbXGR1+x7naVz2bKM5aebqLrROtJCfvpnVeTyc3XT1GnGl1xHNU6Q632Br7yj3qwBY9l+nG/a9fNmtd44Eex1LZvWQFxogP+tE9kGpxDfGF+CPA8WTU7bhoOF0Wd8rSSqo4rVlVwNeTgUjXzEl/OPspM4dcj/nAHtBbq/JasgaupCDsOGFmU6kjBKEwYUzrKwBq4PYzhcOLTwNlou4QgRfxgwGAxa0MHHz8fvw6vLbxcnx1en78PzT8cfTr86U+eqbGDg6a9fTk+w/eSX05O/f/l8fnEVAgshB00db+DAf+7R+O3B27fvDucHB+PDw+QoGh0evXlzeBAfjl7D57ejNywZReN387ejg6PF6/jo3SGMOXj39uCAHbyJX7sDX63z6fji/Oz0a3eV8Wg+OogOj5Lxm9G7ZPRuPzncX7DDt4dH4/2jt0fzcZy8eTeOXr95HY1HDNY+eruIj6LDORu/nb+LE1wFr2l49vny9OLr+Ul48vmXz5ew0OdvlyenIVAgPH8P68FK46PRu/GbcDTaP9o/dBVyH04vTi+Prz5fflV4/VI05TKLqsod8oZPafJPaMvZ5h+H4Vi2nqUlW2SbkAj/urd13APhKNyXre/PPhxfyC8fjj+ef5FfboE107y63YRjteDtXXqLtyivw1ULL7w7gn5/MHh/enb87eMV0ODi7PzDxEng4l/D/R/iNZjB7h44lDRfN3VYFkXtThz3FeejV9QqFyqaum/MfVHewu16pThSMiPJ2xBlsAShRFiIlxXAXBQ5E51SoPV0VQw66zQOFYAtA0gihXm0YogjiJe7HORL+P17FpVL5trDQcwLBRLW0RKnZBHc6PE8rF4fzMP52/FtGDXLki3DqmjWnfk3UbZAIHFawTSYj7dJblaQQVz+FqnP96A9PkbzV13ts3eqE0vMr9ZZShSH8eveATUolxWcQf/6ICIXi4yFFWMJjDh4vaV/Dn8YfE1/RzRBQY3ag/keVnGUh7DPokwq6D0cjVR/CWxZrFr4Y9HcagxoHAWHojUBlrhPYcK9XOn1WOtC1ZTQ+kda67KMgLnmsA4uLntIdUooIEMkOqxqiNIaLarojgEnsnJDQ4eDR5CNCVugdoaZISo+D/luQmrDd/Z+cgCVCc1O0iXq2anUSAGf5PnUe5+i6oVJQbFmueeWc9dHAXwD2GaMQ8D/Fmjl3DT5rZPmTlqz0sui1TyJJmJkAIeYePuj8YGD5sH4wB86c9f1FQSFS9CsgauYR/A4GsDIcPNl/w37zj8BkmKjSAHUnNo2h4D3BhXUBKUBbRrvFF+RtrQG4yKvg9VtkpYe/1JNia5g66RwV4tb+tpPinsXhuVxAUJrOXWberH3tp82iFeQgC72BEJDMQTnV2hFRFWcptOzKKugDdgHEJmO5d7ijAF3oq3HPPpX7ac9xHThUBfYP7RJNF88uLU5GKNgtfJ5Q2cBq9c+GVj5OkirPMp5l68dhKA12DraN1hJjBRYJWkVF8ByQg4Si9EEsBKzNE7rCc5x/qXJMSWFqY834kSUHVoT2plo6IWcE0UPbRjPtd1xu1KLeowqJCGTeOpc42BPDvJnHDaQeMv4CuwPlng0S6HqB6AXi7kn8fR9uTzYexoAjX5RWjEHrZKLoj4rmjw5Lcui9AxOX7gnZDHC/uEwHiT0x8A5rms0lNOa7Bzn79zmec/FLJ4r/knBBJM74zzptuA5gquoBnVSTRxwRWoiBerDa06FYl6x8o4lordu1hnjehP+mWkD6VrLPeLV7tuwTQtgLM4P1u1GczjNwR4zrzuSXpNTLRi/HSfRDaI13LzE85Ab1bihAOT7OkpCVmTFPQgj35lObb6SXSaSgm5yKQsbsVVJXOvMLxvY4Ir1HvdFoZ2yBAC367cmRQdKOjQPFpLAEJ/bw3qQhHi0T5s7KQBnKkFfj2YSY1jUE62+85Ozr9Bel2BreAv3U5PVKfAAuFY120tR/qQxGP4axvgJXChk57+B04gex4Nc9dE1pLRsFpKitXxu2cYri3vbTKOLrbFgEASziQ5PkRLPHTEGKNdgy8xZ5s58f6jusyYqaUzrZsK4HcO4riUTZuc4dHPRVCrjJwZG4ESl6K6CdG9HStHZIhWiQyTsDo//6SVOK+Y1YcyHB0tWe65mFvooIfQ+AxPqdUGr5sV97kp8+HB+PNuR6D0hOgNgOjoUbVV+NuJCtvtFt4QHDEKkpEvMySFMnRHJ5p3E8bdzBYHRTqQFs+WQNFy1w9/KS9pwnQeeM948AIsV+NrPZANyrwL6V+hAmo0WOFC2uyXsABul0+EqHaaZCGrOUF+paxiogQF6IKwEuJcffnafAuqhcKmGJGOisow2PUYHX5jsq7QIfsYJ5581KL7/sjWRir6hqTTsiSKEFFyMtCLJrizEF2J2LSDN+lHcsjo5gy9bXFuRZs9eSBLU+M8lPG3JIH/vYlz7XW3WQvUt3G9g167XZFCp0BwFJWsYBZoM/+iAH1tpBDeIFG4o9YYnP0ycdRKgIXRWoiGGV4LMF7RZpHwU6lR6/sKfFRE88LSUmys8pxVItRBN7u96O5dfWoNSJFqjJjm0Vl1AaM2GHODtj9xSSytSp60FKncRJOlChJ1bGgD5s2aVV+oKi+m2LfIPFETyOOQRtFFVYDi5rAAIZyJaHhULccuBz/Gdv0zJb9650qkwX3ggWi0IWhGu/hKY/MEAqi2GOqTtaNU7qQY/AKfX8yUKOzBQCJBfkm06UWp+5NVz1t3X1333wnXfyRi4uaAKfLfBN252ivPNivi6H5mhzoWzICmLdZg06AOgOe75QV3gdRCOe7s9bUHYBY3oiQT6O/ZmWbIfWnUOaoWVr2Ien3M4t4pgv50J4YkHYDKFTseIjcG4rCsyYZ8kxiyguzzfeBpR4LjS35kgwDxKwhbiA3cx8ah5G4lfbBwCd/nk5+A3dHE0ggH5eoYLLnyUFFYr7b4aGKDRjKFVU/EUQQSL7lvcApRSYOUlAQuNHFvboub4wwdEvyVeXYSo/tBi4YEt3/AIwI30ECDnbmyhb7t4/JO8zXGxWhcVSDOOEuZoUH81efpbw0xnQFFTyPd1yTDUwmP85Gd7ypiZ6KbHUHNtinJCQf8Ag9/vT66+oD/zVXQNNWsIOYAOWkQMuNnKsxpXLAcRO7RsK6E7yCQB6ubrIKroC9dPlsKD2ai7pjCMoimvx6gS1xtwMV8548PDYETQaloKwPGVycKjdIzHTZ9gzcpVUzNvDBw1RElD3vGyKZpKMPCau2xDp9VfqCLElj2+gDESus8C2Dkr10WGATTl5wlQymQGKk09RbGhRj3NpkXHYurO0yzNwWTTtBno6GUO3FnC4VYicKU6YSMwIBKxNE0cldH9diT5joImr4CJGNzjkf//jrCirCc/7nE+pAzTxelV+On0+ALP3mz9evVe27GHf144kV8fzj4g4b3rlh8A2AyZeDUFCinukKFQcNyjUosUDp0fMKhYsyWDSzQvCnTYiAJ0QTDj+i8eGWwja3zVtAr5mYjInxINPNY45X+DhGHEChRQvG7gXxALK8+QASh2OQihbxEV7u7RumZkEYVLg/Yfw0Ru5S1Kkt267aclbCcyqAlbMe+1NAmrJsMYE5G5F7YGbdp+8rXZ1+4KfTuUz+G6LOYRMli9QVUkNkBwrt3uiABner5BDg5U7JYyqCAQkQMx14J7wCSyJ/hPdKio9kA3FyyytDgjwfCkhRDUB030QBYfi1EB/Kav1hfEE/jr0FpZRTOBHtCJof4wru48HV5rj4jGgK3W9aYLm/d2rHmVPyOjA9xaaaMbtrlpkzvdA3mByS0R/UMW9xdBEO1ESU0+x+oWC1/bewabT5p7CdyzKN94O9V1Hw7S3mghOeYi/EpLY7Xtm29Et2acYdaPHCfPRtRvY8fCJpGk7JokCn8Jv43v2bs3HGk+GplWM3laNAPCy3Kln3NImAQDiOSDWpSZPtCfR82hl7FajSoKBbRd6dPMTJLRwSPW3tMcbMWm00Un0sjBzciG0/skau0AE9LTdr6y93uoBDxLAWUnqh2DOEOxv+kD/6vZ+KatLyPPlyiA8EJQdo18Q8Es/iMobRQlG9IIGUNCa6xsmpmSw/A+cxJUXbYEyaSYFv0nkd0TYdAmhy55++GiLNJliCkuzNlWneICntmCo8c/W9UPBwOjiO/NeoX2wnfWGmgpE+hpE6LWOCMnISyFJolQdEd3UQrsZcnvbpbCPXZOvr0/dj58+YZBgVYOOqc5zlYJKBwQxTEYG8SoAcZryHZCtYrJcZ6Arm/gMD3uZyGy10b+fCaEaMLu0pi11jH/6rmIvDxWOIwkRXlA+TgJTK/SmCkxUzVzGEyG8YZfLVGWgPdJ6Hr8qLMPfQVS5hgh02+aJ9d+ZcD1n5Mn5mdu1oHQ4W9JlmpnTPFAu4REt4Ax5yXIoNWz6GkIkQigmg88vWBdaybw9iIo3eI16lRegrpZ4PJCxGXKKBTubwDmQx/qVl1VH+KWDWKgJV01YwIoiwr9u9HQ6VoZ6A0CBy+ZZ4StpBcIB+X3xHW2xzCtpUEhIpKK41rGxc+yhqXFCos1OWmoSuA7d2Q6bNdeLDQuPYunhrD6OgTtRMWKUxeMdgB3z9LlTV2FWC4poWrCRYslK3AivE45JYvzQhgM28d+V2ppNdGWSCpyrMJokVYQ24LdI7D6TsV1jsqNsp9Ud28K7C963M2lBbCwy7nZzEGqTpzPa5affDz/4pyA43/Bfq33fv31I9ZXOT8670+uXn29/ATmEyhEXhh7OOI6zt9pD37LW2NBr3TVMANT0EL8r3r3X9tQqXaerXaxt8w74FwfHo04Z2euni+iWqMh1hrxmJAuxbURs51GZ1sprTnoTlKwSiXOKTgoa2811KVK4bxkak5SM33qUx8s9WSL3oNhfZhlgH0iaWiON0u/ehWaNaNb/yVn2T19M9vyMH2SaOwbbxaO6ZP0HnumUVjWd77W+P7qQzmxr3crhE5BYgeKNWIrJKNUUmflvhH+NiidAscOPtYIC5//cK4kK5sXu2TrLIqB6ckoEpICjVQgTw3OgXN8V4CHEeUWOHwZANMqkGdchN/RujwqCiD2uJyGC3Wfo3BHA7y1WYOec87B3gWrNyezlwotrdiXLCrEEVvqT1WqSrfd3Z+bNMPaN01QY2EZwALMsR2v+aIsfme5kqk1qLrbIAhc7fZiDKs9fmmRyYjNHFfhzOXpd30ojEhfWaEUSlEqiI8mjeGJddQGr3dprJmGXIAVvl5H8sIxRiLxrmUaRW0vWTym1jWMHarsgXFPWWQ0mq2LGIGa4pszO+8bOnv7eiWCW6zBwIerjMWzbL1lrjWoAwQfpaAnBVIkwwNJQjD/mzKKsciUR73M6i4L/o75ohDRc+GkXa1qS1+eM07YcsY8im/nwJLAxnHWJCSRMcK1G4mnoQwdzdqxkejMw8Pt8qqezVWVqJpRpzkaZNcpLHGkO+zjLN8Cp3Tudrj8mRGHuU0J2oXoyKymkTzcMqHLt73muKIe5cBD8qvxuJ5+t2AByJuVdAiMmm1jAFUSUW9PJ8b/oPNdp1OZ69CtvthbJ9cx5HTFInWtOokii1lThXpFkyYwdd8E5Ih2xNxfiDGwiF6PJ8wemdqRkq8v06W4QLMlprqh1rEzTE8MAHeGSwtDL59S9kNnuGFb+L2+owjv6dHCbuivE2G8ZZh/e+hkNycURuxJccp1HjuxcqyH0q+Jcvt1v0prD9poL3hXVkh7d4jeXnrYyekPtSA3htunJLdac5IX9JuUbCNe3BE1awUwc7s9RhxVvLYGPNO2IsCXfpwRfOcZGISmlwyUIN/IMg9Fuvmhs6H2ZIzI5pbDMcqftsZ2WyfR2rgmWRXlQ6wmQT9/a1h5x36vZ+K6xVGeU+x2JD0kc+vKyxE+uWQF8dXUPHqUo/dtzMySq/TwZdo3jT+J6YyX72BM9uk+k9En+j17EJ8C8TLG2gZ4NcaN731nY+OmPa7ZOVl/hNOPJmi0JdwIPF18F2giVxd1lE3NQIwZIAcTNxVDLHaxRiasiqeuslzBxuYPaM0nu24fjhSqFilSaXuTtWnXki/aWlxkbUH3biBecuKPU2e/08mFYqdE1+8MBP7FseIebWNla4Zc+ifT2++8ierLH7SHD4x329vbKfjvCBKUGVNbCVzD/2fd/VH54HRryWyXItzCJ//eLKLYVoLSu+Swt6sySk96J3avgxFEGW7JjGzZAZJJfTOqJED+edwZGsLx5+E8K2J8tKi9XdL/A4PKUCfmDUOuxte5MegRATWkUCePzsvApyhrQTN+/8jfxl58foAzsGqLg+gdTKJZvZcOcriZ4qvnP4NSdCdhZ/1smhXLlFeWaX4kEbILScsWt+l9vtuqWNRwMzwOjW/e83klBjhP11jHM+NFEF2wenpTZ/9deU6FkiHK5NuU3q0+bL2n9kITjtNw+wS7cHZipBjNXdhj+xi8v+x20kMTY8CzQKGDsZAKZ8LTkZPRUfLo7pjME7B9CMjHJU9Nbt+9kqchbZx+WJzJXctR6MBVyeBd9O5/2dIBpiWTJy99AdFlICMAuAWY/hhlFzAjKj952fOJXoB63fXkxc83uozVzDEPsx2SGLATSB7VKTjw92lSyxgOLyqklmfMvKF4nzGVN+2a26l6mehSbcdEIWjA+tZvhi4Rf5IiUeo09Q56tpsS4I2rx4MYd+B1aq3cuLaHzNqqNa7Idl1F8Hb3wWIpiJW6kPX+ndflBki7E5IxYPYUTuMncBo/G6fxUzg9BerxGUrUNBoD+FrcMSwL7tWSZKfLlMt+r0HaZ4c7/2mYmtrb9Rm90hz1q3Cj6MyG+UTZh8iybiWx4b7rmdXeGd2dYniH24qmzckNhT7C8No7w9HGMoY/aY2blnh7RHFWVExGmfRisOmfouqWYhprEUn8rTT2n/K+VbGege11n9dALhe6QeACtaV9OlhfM68E4CeS8GKUSsIbJQrdnTzvOfAnKuZNRMWTIpf/+ArjnJYzKpZ0okXNBJvwB7icYR6Ff1gFjmld95ch4s8rCZpSIOUeXFgKNCedV+SdO1y1z4CQYbX2P1SteFxh5QVMlxWLGppJQbcEf6yJMkltNSh/ZSB+6YpTRnrrgscXaW5G6v5gTFDFArv83L9GP2c/VVdr9Dd5lua38rqqH6bapXj5o34QolGGAUu7ylnDUqt1FgkGFb3mnnF7oMbh6q+xZmbBCwkL8UsC1ssK84cE1NMX45XNpPtol6LrFjoGY3vbkNv3nf+yejVLGUe0X9V1nQ0snWUiIlkeOZ5eziiSdV+kbWXuhXsu60rpWORvsSFl1KsovbxUDCC5B6dOjwC0tYcWvdBeWuZgA3CZbHn/5nH1upBd19HwRXqeVP87EiXyvx9+sNmWQ9M51pz1aAkrtaaAYKsMkwZbJnWsBU1oyFJCXWB0AXQUg5EiiUuAz6u/jTCMq9KjWhK2mzu1HpqC5IdLrzcTwbc0q0oGu2exb4Au4jBqYuP9KoqXZReEti8jWbBw+V4VfdqaYJk/7pBOVisbrya6ldMGEfWXtNj+VBHQPYicWsPqmentXl657jkgQD7NrZiVnVKl3Pj/KQrR9ydQQGfLSIIacq/angRu87E74vCulgSKpBv5xNvebRBECvcZr3Q7EMjEsjDWTRlrgrzRwpgKhXUV4o9HYhabO1/YZE2ssTqwDp+XU9ZS/S8qBPlzuXj++3Xil9iEnA7/dIL+z+THNeHRWzPRJ2uFiSOrJsTXFwLh0gNYZhWVGwmKGncAEiVmslyDvgmzi0rSzTog9UupQBRL/lhxLy0HxQtQXZu5RKGkzCbiHPO3YGFjcyxAs2eq36bV5275XT7H+v09tT6Z9Hm2MX5zroedMVTWLWPp3LF/E+f9gYvwqGkzwQx4PK5gIwyI809UCA78AC309/HJAh/OBC1f0jdRmYbHDTOEB0KvGVQmn37fFaQ13BUxkH7bSkD2AZHfU6zNwvJUbJnKHu328NXEbw7dMXqViED0lf2Xoq4X+v1PLv1Q6WJ4aA2Ty/Cj+p0D4ZcJcetPXL1asP0NvMpribzzx+/8fgzgXml+hGf/uMOPZKerAWS+m5j0GnxYAkPvVpTzZiHwiQwcITW27o3Y5WU7u6TXoJXz3+dfJnAA5qm11UP8t4g7hUMqZ0cPf6jMIYwjuIfmC1x+rIPBAByckILzYUgZuTDE6EgYupP22Y16A+UP/hfNkj7C+1oAAA=="""
MANIFEST_GZIP_BASE64 = """H4sIAAAAAAACCp1dXXNcN45939+iSpEA+IFH1yST8VY2m8pUzatK47Q9mthySnayk3+/t2VZaooAce5N3hISDQIHBwDRTX06vT+9+Xz78e763f3H33+7+nTz4bf3p+vbu19O/7l6f/PP0/urd6e70/3N54/3V7cfbt6dru9uPpyu7k83768/ffz9/s3p6ub+zb9uP29ifr8//denf93cn365fvj/96e3p/vT3bYkbf9e/J+rlFJRofTNv387vbv64dXr//3x6i8fP3w43b+5vXnvCMkvhLz+4e//+PkvlDJd/7Et3GSS5szf/PdP333/Rde70+ern27/8+3t27eOTJoVk5ao7VKMZyHbP5lzS2dB767efHzz8er7Vz86+8XaX1pK33zeNL/6+dXrv3939cPN58U5yiwiV+5l1zmqpQfVWnA92iyCudI+PbrlFKa+S4iaTuFcSqqIU3Kyvdo0EUMCsnWMrPuwlQ2Ecqa8U4oBUcpM8s3/nf75GyxFLF3OIbdLl2LGXNkppcZskFh1FxvkZh9wH/Jyt5CTWTIeR1kth2XaZyQyMEzcSfdJMYBMjfs+9iaLarXvpFpii2C2sNonRSzrapJ9Uop9op0+AoCca9sHZGqxUOm6M1d2h1JrkYIwIqllsLSTP9gqJhKXfRBgk52L7tTFAHWW0vYBidnSpW7Z+7c7XIhYJYDwTlVMcuZW90mpli6t0z4pzZKS97qoW+xV204pNnRlHwdKMlm99D2OlmyGYWmldoXKTTIF0Fandag0ErPgJdk8Ayc4MYveJFJ6IUgJk3236m6fR6pFJrIz40tMt7zx5D66FQO4WylSdx5QzYDeauAdkCvJznT7WKFYlEtS90ViMQviIvvqiGIWxEX3JZFiFcScZV+7U8zmrco+tizVZCjaKSVG8pbut/y2B8nFbOi2RmqfanZHV/r5KgFhjGqy7+4m2aqGt+JxpxQDxRt/l30oriaKM++LqGpTcUrtSzzEhjWLhir72oxqUbHSznK6NvMyZROE56babXLYR3fVxitttepD2RCatSXbrPuQ1rKV4b7cTO2Q4tQMXVuF0nWzL8koizSo6GgmSoWK7LidMmsG3lmXNpNpdSdQmwVU6mmnlG5K6fug2tTKQH0nM3brxozazszc42teLqnsy0CdbJftVM3q01R2lpxdwgNuqGbad0CTgkvfR8HdRDbtvLLszY7UTRk4UnsHUKCc9xnJgvnWbuwLOU2xak1b36WaWlUFt51soGQlrraz11WOa0Dda3sV+zZ8H8WoSeC68ypSrUqDy84baG12PUqqAvXhapYZbSd3q3lTXNjrYJ/Geddvb349T+by1d8+/n7/7v3Np08P5sz5oTWc/PpiY365sSVsI73cuNkM2sjTxg3YyEaZz4htLNMn1g5trC83bqiANrbpEwv2if3lRlLMqjpblTAATNA5N+HQzgk7tWMOyRN4qICfOaFHC3jOCT4kYIhM+CkN82aeAJRbxXZOCKoZPOcEoSqgVyYMNdCfNNMPiD6aMJQUsxBNGCoJC06aMNS22g/aOWGIUQtNGJIO8uyEoQbyJU0YUgLPOWOIMcTThKEO2pbTfE7MQjxhqIDaMs3xCX7mhCHZehlo54Shc90D7ZwwlAnUdsJQVVDbmYfQc04YYgbPOWFIBfOnTBiihH2mTBjqgjGY0LwTrGd4Rh/GYCKzP8FzzjyUMb6VCUObutjONnsFtG2fCzCsHJI5lwm2s8w8BJ6zTBg698rQzglDQphXysxDhMVnmTC0teLYzglDCtYmZa6HwEq8TBhqIIOVCUO9gm3DzENgU1UnDKWO2bZOGNr6YmznhCFO2DmrUQ9h6KsGhkBtJwwRyH21ztkBi8865zKwSq1zLktgBzljCKxSm9GXYV5pRl8G7jT6MlDbCUOlYxZqE4Yy2Au2CUMNrDDaXFMLxiatzXUfaNsJQxnsANqEIarYzj7nMsU4oc81dcLO2eeLIbB66xOGWMFzGldDWGT3CUPnb6BBO+usLfiZMw8JFmW9z/7E8krX2Z8YEjTNPS+mrc71ENg76NzbZwxDOmOoYpygcy4Da02dMQRWGDpfMII1mM48BPYrOmOoYXyrM4bAbvkMof+5/eXf296705//KNf0+GVqTddkf4PJuKa2JKScrhMkgTwJkkEd2JFA0q8zJEFcOxAooTgSpNI1QxKqZ4fUQDs0TwI10Bfdk1AZ1EFdHQhFlAdKqgV0Rs4uIirojezBMhcBjZk9XCZGMZHFDS80OLKHzK3DQ83pQTOLorZobnygyMoeOLkVVISHTs4N9Ai56OyoOclFJ86ZHjqloWRBLmt2NFLJQydLQm3h8qagMUIeOlk7epDmikABTt3li4yK8LmzgyLYQ2fqaArg7AZ7Bp3KPjphES53bvSL4YLdpA6nZC4uwDMqwuVOQkss9tBJBeULdrlTUb5gnzsZtIV46JQkaKHmojOhuBC34EwK2kJc7mwo8YmLzs6oCBedDNe91U3LipqzuQdpqBZu2ZnQWks8dOaGxkhx0dlRXBS3G0poEigud24tGeaRwu5BYFu4dWdFw6wUN0ZQ+i1uZmdBPeLWnQp7pLsHqagWLjoFTcvVRWdDI7W6dSejBXR1u6KK9iPVRefGWuBBPHQSFVSEX3ei9Fv9urOgTnUzu6A1eHXRySh3VnXrCxSdzUMnFzSzNxedDcVFI7elQT3S2D0IoyLcrohgLVx0FhSdzUUn3HC3FTpBj7h1J6EXa00XN4zYQbrbFTHKWt1Hp4IH6f6NEnqv1V3urLAtxKVftKXp7o1SJ/Se0a074UjtbSECNKeHToF7s65uHkHrC02LywfsIJrdKgctDtRFJ6HEp27PntFOQF10JpT4tCxYCzyIy50Ce8TtiggtoNW9UYKv0tXlzoLa4gzOv97en96+//P69XmaxF9NcZZw/hUGMiGyJFA7xykigRwJqZ4LcEQCexIeZkyIBPEk5PNEAZFQPEtqASVUz5IJ9UVzJAgnUIfu6UAC6qCeHR5IF0JUcpXI4DFydrVAMZXJdUcFgZ09XGaBD+IBk+Hwyh4yv1yUQiI8aH4pWCERHjZJMyrCA2dOaIhlF52lgbgglzI7g1qQi05BY4Q8dMrDtRwkwkMnp46K8NApDxcfkAgPnds/qC08dHJGg52ay/+E2qK7WqDoJA+dUlFzssudFaUc9tApGY1UJpe1UMphN6fjIjx05tLRwsJDJwlKv+yh8/y1DfAgHjrPzT6ICw+dqSh6EJc7t4SIaSHJDTNFCzUXnQ3lC3HRCTtVPHSKopEqLndWtL4QN7OLgNASN7NXQc3ZXC1gj3SXclD6FQ+dqaPmLG5mfxjEQiLcbqiiObW4dSfMF8XlzgLbwq07KxqpxUVng0X4LRFadxaXOxnWwq07GcaFuqUr2l9WD51ZBbRF9Xt1NKdWF50ZDfbqoZM6WoNXF50JJb5a3NIVzWa1LkpXUAu3Y1e0dK1uy95gW7joFFSLltyrB5T4Wl6UKOD9h4vOjqKzudxZUXQ29zIpdfQgLjoZrS+an9nRorG5mb2jzURz0ckogzc3sxPK4N1FZ2dUhMudlMCDdLdnV9QjnRf9CHjD53ZFcN3Z/a4IvW/tLjofvu0LiWiLa0LwIG5XlNFOoKtb8aGspW7d+TBRhkS43CnodZCSW6KgnYD6PTvKWup2RRkW4d/Eo4WSVtcWKGtpWxSNoDn9uhPlC3XRmdHLygmcT1/Lr0cnRBffe+ZjE6KnQS56CvZ0SCgmxJOQ0clI8SR09Dqqer4QtDVsjoTMaKPdHQkE33yo582OniK7oMyoO7OHylzQ4j27sISHCdnDpcC8nT1gUoKHTC4yM4rM7EIzoUVa9rD55XtqxyZET2EOT8tcdJZ6eEL09GVlWISLTm2HJ0RP3xyEaddHJwotEjfM8uEJET1PlNPBCdHFDwaPTogufpKVDk6Inr79QocnRBfmpIMTInq+ljs6IXr66goswkNngssjdtEJj/1Y3DyihydE9Dx8TAcnRE9fSoVFuNyp6fCE6KnIQq8o2UVnInQS4KMTngR46OSK2kLczN7RGxxxK044UsVFZ4fHO8XFBR2eENHz17PywQnRxa938sEJ0dMzAunwhOirCIW/4+Whk+DLweKhUxh1anHRyWjRWFx0MnohVtzM3tAYKcXtRtC+rrjcqej1S2kuX8ADN5c7C5rNip/ZUeKrPjr58IToqQZHv4NY3czeYRHs2gKezYgbI/Ccyq870WxWPXSKwlr4HTt6y1m7GyPobKbq4gbm6ITo4gdAdHBCdPHrHTo4Ibr4tR8fnBA9Ndz98IToqROA51Ruz650eEJ0kQT44IToqVBCWau5mV3b4QkRPX9NIB2cED2xFjzecTN7R7uiTotOIB+cENHz8JEOToiefmeHxkh3M3viwxOii/YuHZwQ0fMkgA9OiOj5S/l8cEJ0cefKBydEw++1j02InnCBZjN1M3vRwxOiryJaPjwh+mpO+Hu26t/Ew1q46ISzmbrcCUeq+uhEL7HVzewVnWu8+IHbxlR7f/2eHQkM/0KZHAki8HsyjoRzm5uOvDL3rAP8K73iSejoU0fVkXD+/mE68srcszc7+ihD9yTAb3WoZwdCHxjKHigFfmAoe6gkhl9P9GDJHf3Re3ZxKfDLah4wKaO/3s8eMs+vifGhV+YutEgHX5l7dir8wFDubojpwVfmnvHNFX0WzUNngp/qIA+dwuXgK3MXlIf+ep94ISIdemXuIszQ91fIQyfD70FSdXGBQos8dLLCL+519yAoX5DLnfADAOxyZzv6ytwzLvD33Wjh1HTolbnnSIWfGmU3qXdYCw+diVCAs8udDdbC5U6mg6/MPWtB6NtV7KETf+RTPHSef2dHh16Zez4I/C6y+NyJRqp46OSKhpmIG2awFm5m53LwlbmLkjEdfGXuAhdocSA+d8Ja6MKpdOiVuYuDoE+NFpc7O5rNiotO+NG+4nIn/GBIcTM7/jib2xHBb44UF50FLdeKi05Cw6y4TVFBnyYuHjpJUBHVzewKP4vmcicM8Oo26wSLYLdEQbvUKm4Zj6Kz+nWnHnxl7sIjcMvvZvaCvhNaPXSmBJvT7Yoy/L69e5GU+8FX5p5FwABvbt3JaBnf3Myu6eArc88iEoqL5mZ2/MU9t+6UdPCVuWd0VpR+m4tOOI80Xdwc8KFX5i5yaj/4ytwFX8DPormZXeDrORedcNHY3bqzoTcH3eXOrAdfmbvQAn73z6870d6su+hkPvjK3EV9gXKnutyp9eArcxceQaGlqxslOvTK3MU0AX3wVGXhkWOvzF3cd6LNhLqZndLBV+YucAG/PujWnfBz/+rXnehc4wzOb//6/asfv/zxoi9/aunNxzcfr7b/Zg2Dnhbz498K9xbT5WIiXi7mYfHj3yD0FsvlYu3rxeVy8RYwy8X1cnF+/MON3uI2mK6vD9gvF4uuF+vl4p7WB8yDCzfjrVcPPqTHvw7orh6c2B7/Uru7mkd8rA+ZBzdyXXsmD36sgbXz4EitgQXbCKi12/Pgyg0l69WDL1NdBxil0YKyXj34smlfrx582fP6lDT4MgU2ocGXEuk9+LJHq0dfljViafBl74HswZelBhYcfUlr2Tz4Muvagjz4UgJ880iuvI5LHnz59W/DuqtHeg28w4MvOYgGrmM0BJq0EYNrhuDBlxzwIA++lCAaZEyTgedl5NggLmWMS1njW0ZfRpqMHBtEgwy+zC3QZIxLXWNQ2phJ1jwoY1ymQJPRlyUoCEZfBnFZRo7N61OWwZe5rzFYRl+WNb7L6EtZW7CMcZmC1WPhE+CkDL7MASOXwZc1KKrK4EtNQb02+LIHGKyDLysFq1/E5dqCdfClBJVSlbF2DFYPvqSghqiDL0tg7zpWsZEF++j5QPboS11HQxt9mdas2QZfphSU6oMvS1lbsA2+bJFsGeuqQO8yNlyBTQZftqBab4Mva1vn4jbmSw1sMviy5DVD9NGXQfXY88iDa5v0F7VPsJrHlm6dG/rgyxxgsI9xmdax00dftkD2WMdKIHvk2MjeYx0b5EsdfFl1zcg6+pLWiNUxXwbMpoMvaxCXOnJsUPXq6EsKTllHewenHHxJAWtqH2M+0Hvk2CAuz678/tUPr3/6koq/gOrnV6///p133fO8+rE8cVfT5er2+Iff3dV8uZoeS013tVyu1sc/Qu6uLper+6NJ3NX1cnV5DHl3dRtO+djAuKv75WrOgd46asKBdwZncrh88GaRQJc8uLNSYMQ8+PMrAfnLXzg0MHoePNooQEt+4dLoqINPtUZH7aPukTKDV7kFytDg1cxR0A1e1RIYkgav5hpJH7xaorijwasllD54tXAkffDqloiC5WOk5iBSafBqK5F0HTkmYrs0QiyQzoNXWQND8uBVpkj64NVMkfTBq7UF4cFlZPYgx/CLWI2ktzGaAtLjwatSAojxyMAS6C4jA3MAAhm9mgO7y4uEGi0fGbhFygxe7REDy8jAHOXrwasapWB5wcDRUUcGliC0ZfBqlSA8ShpJL4BYGbyaelSbDF7tHFimDF6V6Khl9CpHuo9e1cCrZfBqL5Hug1cpBV4tY7GUImV0zHxRlTd4tUXEUcfatwdMUMe8GiXKOng1RRCrg1dziZQpY5Uf2L0OXq0ahEdtY3gEPFP7iMjAq3X0qkbV+FgtRVm7jV6N8N5GBo4Q2QavaoqUGWM1BV5tL2rgyDJ1LH8iZUYGjnJT6yPEIsvoWHYGeO8vGDg4ah+82nqge6ex7AwA3EcG1oBn+uDVFPF7H7wqHPWTIwNHtVhvY/YI3NRHr0YA7jpmj0AZHbxK0VF19GqLltMYfIFXdayWJFo+MjAHltHBqykKPq0jCKJrgsGrKWIx7SMiI2XGaqkEdj879debu19u7z79+uc1ffkzlzzc/vxw8/nb27dvrdulaWcebOXsJGtnyT3eydbOPhjc2SnWThms4+wsprY1xTuraaEh6J2dzdpZE6Btt3aqAP5UU9uhWPGQYIKoI+pmE0UVwp8JIyoAGLKJo5oBn2YTSMoA7LOJpNyRs5pQSgKETLaxRMhZux2oGdhqoqk14Kxko6kBQKRsQwL5VBNNUpFPZds5QJyT2GcFnEMmmliAoCMTTVoY2GqiSTuy1URTQzBMJpokAZ/KJpqKIltNNAnC32yiqSUATWznuAQEHZtoqgVAE5to6op8qp3mGPlUm5sagGE20ZQUcY6JJs6Ac8QulxCFxeamBlhYTDQlBjAsbKcrIOhE7HSFfKqJJlIgSYqd6ZDiUBw0IVvtsgnyq4mmjiTJkmyCASqYYlffDXBOITtegcgpTt2EfKqT6RAzmWgab5m9rdWmNaTsN9GUBTmrgyYAiEVt+CPdhp3pEL9WE01cATNVG00IhquJJkI6yGqiqSBtTrWrcIT9a7XLS8TCJpokIwp3u9BDFFa7dwUs3Ew0KVI3NRNNDanCm103IQTTTDQxUjc1sZsVIOiaUzchPb7NTUg72JqdX4F4bSaacgLitdk9XQbO2u2ergBA7E5PB0Ci2xdNSFHbTTQ1JHK6XYULYOFuZzqka+7VDnWAJbqd6TKisI2miihs93QKBJ0mOzUjl1x2pkNyjtpVuCAK29wEndVEEyORo3amQ8oQtdGEXGlos9MV4hwTTYQEnapdSwCf+gCmP25//Xj3+XT3+frD089xr/+oDx5Gbp5yIKQi6ZoCIQ0JSQ6ElI7eli+EZOT6sETHQVrJGghhxDstEKKk4M36QggResm+ECLQpXkKVRH06n2FWeTiKEegVSTd5gi13DN6N79CHHSiCLcVucbLEXAF6V9zhNyEVF+5h+zU0Sv9hZSOTNMowm5FameKsNsIvvNf0Rx0rx1htyLlEEXYTdCJIuwWpM+lELsEjwpW1oXmMiHtIl0PaejphM4SlrlV0LHCSpdE6IRhZRdkKs4RdlNv6NxhWUJldASxkoLcvXGE3YrUhBxiF7lB5x7mAELHFavciIxaJMRugYcYS10qOs9YYRfShUMfJXTKseQ6QQceK+tCU4gaMmZFxyArKch9gkTY1d7R4ciyIqvonGSFOiSOSoTdgvRoJcIul44OUlbWZUJnKitP54aOV5a1N6OTllVMQ6OeFvaMGZ2/rCpVaMQW9moVnsqsshrCUjXCbkca8hr2akhPUsNeDRo5hTUDUr/UkHch64b1LjQdCrGLfDWththFrsNqhN2E6NIi7HbJ6DhoWXvDk6HlbQWhQ6JlD1vQedEyN8Kjo2VMd3SKtMwBgg6UllUQobOllRTknqFp2MNmdOK0rBkEHT4tmUHQOdSy41N0JLW8CiV0OrW8ZE7ooGolBfqucg27z46Or1Z2oYxOspY+auhQa8UMouh8a8V1BI+6lvzC6NRreXee0QHYMo46Ogtb1ZhINGpc7xI6IVvpgnx7RluY1Ridmy3vVAkdoa1i2jzR/wO0/FJ/qPEAAA=="""

runtime_dir = Path("/kaggle/working/aide_full_commfor_runtime")
runtime_dir.mkdir(parents=True, exist_ok=True)
train_runtime_path = runtime_dir / "train_aide_full_tiny_commfor_kaggle.py"
test_runtime_path = runtime_dir / "test_aide_full_commfor_exact_kaggle.py"
manifest_path = runtime_dir / "selected_samples.csv"

train_runtime_path.write_bytes(gzip.decompress(base64.b64decode(TRAIN_RUNTIME_GZIP_BASE64)))
test_runtime_path.write_bytes(gzip.decompress(base64.b64decode(TEST_RUNTIME_GZIP_BASE64)))
manifest_path.write_bytes(gzip.decompress(base64.b64decode(MANIFEST_GZIP_BASE64)))

sys.path.insert(0, str(runtime_dir))
spec = importlib.util.spec_from_file_location("aide_full_commfor_runner", test_runtime_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)

CONFIG["manifest_path"] = str(manifest_path)
print("Runtime:", test_runtime_path)
print("Embedded exact manifest:", manifest_path)
print("Full AIDE CommFor inference runner imported successfully.")

In [ ]:
result = runner.run_inference(CONFIG)
result["experiment"]

In [ ]:
import json
import pandas as pd
from IPython.display import FileLink, display

run_dir = Path(CONFIG["output_root"])

print("CommFor overall: 100 real + 900 fake unique samples")
display(pd.DataFrame([json.loads((run_dir / "metrics/commfor_overall.json").read_text())]))

print("CommFor per generator: mỗi hàng = cùng 100 real + 100 fake của generator đó (%)")
per_generator = pd.read_csv(run_dir / "metrics/commfor_generator_metrics.csv")
metric_columns = [
    "accuracy", "balanced_accuracy", "real_recall", "fake_recall",
    "fake_precision", "fake_f1", "roc_auc", "average_precision",
]
shown = per_generator.copy()
for column in metric_columns:
    shown[column] = (100 * shown[column]).round(2)
display(shown)

print("Macro summary")
display(pd.DataFrame([json.loads((run_dir / "metrics/commfor_macro_summary.json").read_text())]))

print("Download:")
display(FileLink("/kaggle/working/aide_full_commfor_exact_eval.zip"))

## Output quan trọng

- `metrics/commfor_overall.json`: metric trên 1.000 ảnh unique (lưu ý tỷ lệ lớp 100 real / 900 fake).
- `metrics/commfor_macro_summary.json`: macro trung bình qua 9 cohort cân bằng theo generator.
- `metrics/commfor_generator_metrics.csv`: chi tiết từng generator, mỗi hàng gồm 100 shared real + 100 fake.
- `predictions/commfor_predictions.csv`: xác suất và dự đoán từng ảnh.
- `dataset/commfor_manifest_used.csv`: manifest chính xác đã sử dụng.
- `provenance/checkpoint.json`: checksum và metadata checkpoint.
- `aide_full_commfor_exact_eval.zip`: gói kết quả tải về.